# PC-FNO3D réel (Grand Tunis) — OS6 : plan de déploiement du réseau ANEO (Colab, L4)

Expériences de simulation de systèmes d'observation (OSSE) sur le **modèle livré** (celui que
désigne `runs/decision_v4.json`, w par continuité), **sans entraînement** : un capteur ANEO simulé
est une maille où le modèle voit la vérité de surface du cache, comme une station réelle.

1. emplacements candidats (terre ferme, altitude ≤ 650 m, pente ≤ 20°, à 1 km du bord, à 1 km des
   stations et entre capteurs ; zones exclues de `os6_contraintes.json` s'il est déposé sur le Drive) ;
2. placement glouton sur la **validation** : 5 capteurs par étape là où l'erreur résiduelle est la
   plus forte, gain mesuré à chaque étape (jusqu'à 200) ;
3. courbe de dimensionnement sur le **test** (une fois) : optimisé, grille régulière, hasard, à nombre
   égal ; nombre minimal pour un pire RMSE de niveau ≤ 0,5 m/s ;
4. plan S18 : `runs/os6_reseau_aneo/plan_aneo.csv`, `.geojson`, `carte.png`, `courbe.png`, `os6.json`.
5. atlas éolien annuel à 100 m (E8) et export SIG GeoTIFF (E15) : `runs/atlas_eolien/`.

Le front-end ATOM Wind lit le plan (`/api/atom-wind/reseau-aneo`) au prochain démarrage du service.
Reprise : une étape déjà écrite (placement, courbe) est relue. La session s'arrête à la fin.

## 1. Drive, code et données

In [ ]:
from google.colab import drive
import os as _os
if not _os.path.isdir("/content/drive/MyDrive"):   # deja monte (panneau Fichiers) : ne pas redemander
    drive.mount("/content/drive")

import json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path

GT = Path("/content/drive/MyDrive/pcfno3d_grand_tunis")
GT.mkdir(parents=True, exist_ok=True)
JOURNAUX = GT / "journaux"
JOURNAUX.mkdir(exist_ok=True)

# code : archive deposee sur le Drive depuis ines-pc
CODE = Path("/content/PFE")
if CODE.exists():
    shutil.rmtree(CODE)
CODE.mkdir(parents=True)
with zipfile.ZipFile(GT / "pcfno3d_code.zip") as zf:
    zf.extractall(CODE)
tampon = CODE / "PCFNO3D_VERSION.txt"
print(tampon.read_text().strip() if tampon.exists() else "tampon : absent")
%cd /content/PFE
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q pandera xarray netCDF4 h5netcdf 2>&1 | tail -2
sys.path[:0] = ["/content/PFE", "/content/PFE/src"]

def lancer(nom: str, commande: str) -> None:
    """Commande obligatoire ; sortie affichee ET journalisee sur le Drive ; arret si echec."""
    t0 = time.time()
    with open(JOURNAUX / f"{nom}.log", "a", encoding="utf-8") as log:
        log.write(f"\n===== {time.strftime('%Y-%m-%d %H:%M:%S')} : {commande}\n")
        p = subprocess.Popen(commande, shell=True, cwd="/content/PFE", stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1)
        for ligne in p.stdout:
            print(ligne, end="")
            log.write(ligne)
            log.flush()
        p.wait()
    print(f"[{nom}] {time.time() - t0:.0f} s, code {p.returncode}")
    if p.returncode:
        raise RuntimeError(f"{nom} a echoue (code {p.returncode}) : voir {JOURNAUX / (nom + '.log')}")

# donnees : grille reelle, forcage ERA5, stations (une archive sur le Drive)
DON = Path("/content/donnees")
if not (DON / "grand_tunis" / "forcage_era5.csv").exists():
    with zipfile.ZipFile(GT / "grand_tunis_donnees.zip") as zf:
        zf.extractall(DON)
# mises a jour deposees apres coup sur le Drive (morceaux de moins de 10 Mo) :
# maj_<fichier> ou maj_<fichier>.000, .001... -> remplacent <fichier> dans donnees/grand_tunis
import collections
morceaux = collections.defaultdict(list)
for f in sorted(GT.glob("maj_*")):
    nom = f.name[4:]
    tete, _, queue = nom.rpartition(".")
    morceaux[tete if queue.isdigit() and tete else nom].append(f)
for nom, fs in morceaux.items():
    with open(DON / "grand_tunis" / nom, "wb") as out:
        for f in sorted(fs):
            out.write(f.read_bytes())
    print("mise a jour :", nom, "<-", [f.name for f in sorted(fs)])
print(sorted(p.name for p in (DON / "grand_tunis").iterdir()))

## 2. Cache (3 parties)

In [ ]:
# cache : les 3 parties du Drive -> disque local ; index fusionne
LOC = Path("/content/grand_tunis")
CACHE, JEU = LOC / "cache", LOC / "jeu"
for k in (1, 2, 3):
    if not (GT / f"cache_partie_{k}.zip").exists():
        raise FileNotFoundError(f"cache_partie_{k}.zip absent : finir le carnet de generation")
if not (CACHE / "index.csv").exists():
    CACHE.mkdir(parents=True, exist_ok=True)
    for k in (1, 2, 3):
        with zipfile.ZipFile(GT / f"cache_partie_{k}.zip") as zf:
            zf.extractall(CACHE)
    with zipfile.ZipFile(GT / "jeu.zip") as zf:
        zf.extractall(JEU)
    import pandas as pd
    idx = pd.concat([pd.read_csv(CACHE / f"index_partie_{k}.csv") for k in (1, 2, 3)])
    idx.drop_duplicates("stem").to_csv(CACHE / "index.csv", index=False)
import pandas as pd
print(pd.read_csv(CACHE / "index.csv")["split"].value_counts().to_string())

# la grille complete (aneo_grid_points.csv) est dans jeu.zip
if not (GT / "best_dso_v7.pt").exists():
    raise FileNotFoundError("deposer best_dso_v7.pt (poids v7) dans MyDrive/pcfno3d_grand_tunis/")
lancer("controle", "python -u -m pcfno3d -c grand_tunis_v2_long_colab check")

## 3. Placement, courbe, plan — puis fin de la session

In [ ]:
import shlex
def choisir_poids(src):
    """Le Drive garde parfois deux fichiers du meme nom (grand_tunis_v4_dso : deux best.pt de 1,69 Go,
    vus le 31/10) ; Colab peut n'en montrer qu'un, au hasard. On retient celui dont l'epoque est celle
    du test evalue (eval_pcfno3d/best_wcont, sinon best), meme s'il n'y en a qu'un : jamais un poids
    pris au hasard."""
    cands = sorted(src.glob("best*.pt"))
    if not cands:
        return src / "best.pt"
    attendu = next((json.loads(f.read_text()).get("epoque") for d in ("best_wcont", "best")
                    if (f := src / "eval_pcfno3d" / d / "test_summary.json").exists()), None)
    import torch
    epoques = {}
    for c in cands:
        ck = torch.load(str(c), map_location="cpu", mmap=True, weights_only=False)
        epoques[c] = ck.get("epoch", -2) + 1
        del ck
    print("poids candidats :", {c.name: e for c, e in epoques.items()}, "| epoque evaluee :", attendu)
    ok = [c for c in cands if epoques[c] == attendu] if attendu is not None else cands
    if not ok:
        raise RuntimeError(f"aucun poids de {src} n'a l'epoque evaluee ({attendu}) : verifie sur le Drive")
    return max(ok, key=lambda c: c.stat().st_mtime)          # a epoque egale : le plus recent

from google.colab import runtime
import traceback
RUNS = GT / "runs"
DECISION = RUNS / "decision_v4.json"
livre = json.loads(DECISION.read_text())["modele_livre"] if DECISION.exists() else "grand_tunis_v2_long"
CONF = {"grand_tunis_v4_dso": "grand_tunis_v4_dso_colab",
        "grand_tunis_v2_long": "grand_tunis_v2_long_colab"}[livre]
SORTIE = RUNS / "os6_reseau_aneo"
CONTRAINTES = GT / "os6_contraintes.json"      # zones exclues, budget... (facultatif, depose sur le Drive)
print("modele livre :", livre, "| contraintes :", CONTRAINTES if CONTRAINTES.exists() else "par defaut")
try:
    CKPT = shlex.quote(str(choisir_poids(RUNS / livre)))
    lancer("os6_reseau_aneo",
           f"python -u scripts/os6_reseau_aneo.py -c {CONF} --checkpoint {CKPT} --set inference.w_continuite=true "
           f"--sortie {SORTIE} --n-val 24 --lot 4"
           + (f" --contraintes {CONTRAINTES}" if CONTRAINTES.exists() else "")
           # capteurs ANEO deja poses (lat, lon) : le plan repart d'eux (BF78)
           + (f" --installes {GT / 'os6_installes.csv'}" if (GT / "os6_installes.csv").exists() else ""))
    # atlas eolien annuel (E8) et couches GeoTIFF (E15), a partir des reconstructions du cache
    if not (RUNS / "atlas_eolien" / "atlas.json").exists():
        lancer("atlas_eolien", f"python -u scripts/atlas_eolien.py -c {CONF} --checkpoint {CKPT} "
                               "--set inference.w_continuite=true "
                               f"--sortie {RUNS / 'atlas_eolien'}")
    print(json.dumps({k: v for k, v in json.loads((SORTIE / "os6.json").read_text()).items()
                      if k in ("nombre_recommande", "candidats", "capteurs_places", "reseau_actuel",
                               "comparaison_net1")}, indent=1, ensure_ascii=False))
    # bilan de conformite au cahier v12 (lecture seule) : runs/bilan_conformite.md
    lancer("bilan_conformite", f"python -u scripts/bilan_conformite.py --runs {RUNS}")
except Exception:
    traceback.print_exc()
finally:
    runtime.unassign()